# Triton Inference Server 教程

本教程详细介绍 Triton Inference Server 的使用，包括：

1. **Triton 简介**: 高性能推理服务器
2. **模型配置**: config.pbtxt 配置文件
3. **客户端使用**: HTTP 和 gRPC 客户端
4. **动态批处理**: 提高吞吐量
5. **模型集成**: Ensemble 模型

---

## Triton 架构

```
┌─────────────────────────────────────────────────────────┐
│                  Triton Inference Server                 │
├─────────────────────────────────────────────────────────┤
│  ┌─────────┐  ┌─────────┐  ┌─────────┐  ┌─────────┐    │
│  │ Model A │  │ Model B │  │ Model C │  │ Model D │    │
│  │ (ONNX)  │  │ (TRT)   │  │(PyTorch)│  │(TensorFlow)│ │
│  └─────────┘  └─────────┘  └─────────┘  └─────────┘    │
├─────────────────────────────────────────────────────────┤
│  动态批处理 │ 模型并发 │ 模型版本管理 │ GPU 调度        │
├─────────────────────────────────────────────────────────┤
│  HTTP/REST  │  gRPC  │  C API                          │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
import sys
import os
sys.path.insert(0, '../src')

import numpy as np

# 检查 Triton 客户端
try:
    import tritonclient.http as httpclient
    print("Triton HTTP 客户端已安装")
    TRITON_HTTP_AVAILABLE = True
except ImportError:
    print("Triton HTTP 客户端未安装")
    TRITON_HTTP_AVAILABLE = False

try:
    import tritonclient.grpc as grpcclient
    print("Triton gRPC 客户端已安装")
    TRITON_GRPC_AVAILABLE = True
except ImportError:
    print("Triton gRPC 客户端未安装")
    TRITON_GRPC_AVAILABLE = False

if not (TRITON_HTTP_AVAILABLE or TRITON_GRPC_AVAILABLE):
    print("\n安装命令: pip install tritonclient[all]")

# 导入封装模块
from triton_client import (
    TRITON_AVAILABLE,
    TritonDataType,
    numpy_to_triton_dtype,
    ModelConfigGenerator,
    ModelInput,
    ModelOutput,
    ModelMetadata,
)

print(f"\n模块 Triton 可用: {TRITON_AVAILABLE}")

## 1. 数据类型映射

Triton 使用特定的数据类型，需要与 NumPy 类型进行转换。

In [ ]:
# Triton 数据类型
print("Triton 数据类型:")
for dtype in TritonDataType:
    print(f"  {dtype.name}: {dtype.value}")

In [ ]:
# NumPy 到 Triton 类型转换
print("NumPy → Triton 类型转换:")

test_arrays = [
    np.array([1.0], dtype=np.float32),
    np.array([1.0], dtype=np.float64),
    np.array([1], dtype=np.int32),
    np.array([1], dtype=np.int64),
    np.array([1], dtype=np.uint8),
    np.array([True], dtype=np.bool_),
]

for arr in test_arrays:
    triton_dtype = numpy_to_triton_dtype(arr.dtype)
    print(f"  {arr.dtype} → {triton_dtype}")

## 2. 模型仓库结构

Triton 使用特定的目录结构来组织模型：

```
model_repository/
├── model_a/
│   ├── config.pbtxt      # 模型配置
│   ├── 1/                # 版本 1
│   │   └── model.onnx
│   └── 2/                # 版本 2
│       └── model.onnx
├── model_b/
│   ├── config.pbtxt
│   └── 1/
│       └── model.plan    # TensorRT 引擎
└── ensemble_model/
    ├── config.pbtxt
    └── 1/
        └── (empty)       # Ensemble 无模型文件
```

## 3. 模型配置生成

使用 ModelConfigGenerator 生成 config.pbtxt 配置文件。

In [ ]:
# 生成基本配置
config = ModelConfigGenerator.generate_config(
    name="image_classifier",
    platform="onnxruntime_onnx",
    max_batch_size=32,
    inputs=[{
        "name": "input",
        "data_type": "TYPE_FP32",
        "dims": [3, 224, 224]
    }],
    outputs=[{
        "name": "output",
        "data_type": "TYPE_FP32",
        "dims": [1000]
    }],
    dynamic_batching=True,
    preferred_batch_sizes=[4, 8, 16, 32],
    instance_count=2,
    device="GPU"
)

print("生成的 config.pbtxt:")
print("=" * 50)
print(config)

In [ ]:
# 生成 CPU 配置
cpu_config = ModelConfigGenerator.generate_config(
    name="text_encoder",
    platform="onnxruntime_onnx",
    max_batch_size=16,
    inputs=[{
        "name": "input_ids",
        "data_type": "TYPE_INT64",
        "dims": [512]
    }],
    outputs=[{
        "name": "embeddings",
        "data_type": "TYPE_FP32",
        "dims": [768]
    }],
    device="CPU",
    instance_count=4
)

print("CPU 模型配置:")
print("=" * 50)
print(cpu_config)

## 4. Ensemble 模型配置

Ensemble 模型可以将多个模型串联成流水线。

In [ ]:
# 生成 Ensemble 配置
ensemble_config = ModelConfigGenerator.generate_ensemble_config(
    name="image_pipeline",
    max_batch_size=32,
    inputs=[{
        "name": "raw_image",
        "data_type": "TYPE_UINT8",
        "dims": [-1, -1, 3]  # 动态尺寸
    }],
    outputs=[{
        "name": "classification",
        "data_type": "TYPE_FP32",
        "dims": [1000]
    }],
    steps=[
        {
            "model_name": "preprocessing",
            "model_version": -1,
            "input_map": {"raw_input": "raw_image"},
            "output_map": {"processed_output": "preprocessed"}
        },
        {
            "model_name": "classifier",
            "model_version": -1,
            "input_map": {"input": "preprocessed"},
            "output_map": {"output": "classification"}
        }
    ]
)

print("Ensemble 配置:")
print("=" * 50)
print(ensemble_config)

## 5. 模型元数据

使用数据类表示模型信息。

In [ ]:
# 创建模型元数据
metadata = ModelMetadata(
    name="resnet50",
    versions=["1", "2", "3"],
    platform="onnxruntime_onnx",
    inputs=[
        ModelInput(name="input", datatype="FP32", shape=[1, 3, 224, 224])
    ],
    outputs=[
        ModelOutput(name="output", datatype="FP32", shape=[1, 1000])
    ]
)

print("模型元数据:")
print(f"  名称: {metadata.name}")
print(f"  版本: {metadata.versions}")
print(f"  平台: {metadata.platform}")
print(f"  输入: {metadata.inputs[0].name} - {metadata.inputs[0].shape}")
print(f"  输出: {metadata.outputs[0].name} - {metadata.outputs[0].shape}")

## 6. Triton 客户端使用

以下代码展示如何使用 Triton 客户端（需要运行中的 Triton 服务器）。

In [ ]:
if TRITON_AVAILABLE:
    from triton_client import TritonClient, create_triton_client
    
    print("Triton 客户端使用示例:")
    print("""
# 创建客户端
client = create_triton_client(
    url="localhost:8000",
    protocol="http"  # 或 "grpc"
)

# 检查服务器状态
if client.is_server_ready():
    print("服务器就绪")

# 检查模型状态
if client.is_model_ready("resnet50"):
    print("模型就绪")

# 获取模型元数据
metadata = client.get_model_metadata("resnet50")
print(f"模型输入: {metadata.inputs}")

# 执行推理
input_data = np.random.randn(1, 3, 224, 224).astype(np.float32)
result = client.infer(
    model_name="resnet50",
    inputs={"input": input_data}
)

print(f"输出形状: {result.outputs['output'].shape}")
print(f"推理延迟: {result.latency_ms:.2f}ms")

# 关闭客户端
client.close()
""")
else:
    print("Triton 客户端未安装，跳过示例")

## 7. 批量推理

In [ ]:
print("批量推理示例:")
print("""
# 准备批量输入
batch_inputs = [
    {"input": np.random.randn(1, 3, 224, 224).astype(np.float32)}
    for _ in range(10)
]

# 批量推理
results = client.batch_infer(
    model_name="resnet50",
    batch_inputs=batch_inputs
)

# 处理结果
for i, result in enumerate(results):
    print(f"请求 {i}: 延迟 {result.latency_ms:.2f}ms")
""")

## 8. 启动 Triton 服务器

使用 Docker 启动 Triton 服务器：

In [ ]:
print("启动 Triton 服务器:")
print("""
# 使用 Docker 启动
docker run --gpus all --rm -p 8000:8000 -p 8001:8001 -p 8002:8002 \\
    -v /path/to/model_repository:/models \\
    nvcr.io/nvidia/tritonserver:23.10-py3 \\
    tritonserver --model-repository=/models

# 端口说明:
# 8000: HTTP 端口
# 8001: gRPC 端口
# 8002: Metrics 端口

# 常用参数:
# --model-control-mode=poll  # 自动检测模型更新
# --repository-poll-secs=30  # 轮询间隔
# --strict-model-config=false  # 自动生成配置
""")

## 9. 性能优化配置

In [ ]:
# 高性能配置示例
high_perf_config = ModelConfigGenerator.generate_config(
    name="high_perf_model",
    platform="tensorrt_plan",
    max_batch_size=64,
    inputs=[{
        "name": "input",
        "data_type": "TYPE_FP16",  # 使用 FP16
        "dims": [3, 224, 224]
    }],
    outputs=[{
        "name": "output",
        "data_type": "TYPE_FP16",
        "dims": [1000]
    }],
    dynamic_batching=True,
    preferred_batch_sizes=[8, 16, 32, 64],
    max_queue_delay_microseconds=50,  # 更短的等待时间
    instance_count=4,  # 更多实例
    device="GPU"
)

print("高性能配置:")
print("=" * 50)
print(high_perf_config)

## 总结

本教程介绍了 Triton Inference Server 的核心功能：

1. **模型仓库**: 标准化的目录结构
2. **配置生成**: config.pbtxt 配置文件
3. **客户端使用**: HTTP 和 gRPC 协议
4. **动态批处理**: 提高吞吐量
5. **Ensemble**: 模型流水线

### 最佳实践

- 使用 TensorRT 后端获得最佳 GPU 性能
- 启用动态批处理提高吞吐量
- 使用 gRPC 协议减少延迟
- 配置多个模型实例实现并行推理
- 使用 Ensemble 构建复杂推理流水线